In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, re, functools, json
import pandas as pd
import numpy as np

from src.scryfall_http_service import ScryfallHttpService
from src.scryfall_data_service import ScryfallDataService
from src.cache_service import CacheService, CacheType

pd.set_option('display.max_columns', None)

os.getcwd()

'c:\\Users\\tallo\\Documents\\programming_projects\\mtg-coding'

In [3]:
def get_commanders_from_txt_file(fpath: str) -> list[str]:
    with open(fpath, 'r') as f:
        return [x.strip() for x in f.read().split("\n")]


In [4]:
sfds = ScryfallDataService(
    base_url="https://api.scryfall.com/", 
    json_cache_service=CacheService("./cache/json/", cache_type=CacheType.JSON), 
    png_cache_service=CacheService("./cache/png/", cache_type=CacheType.PNG), 
    http_client=ScryfallHttpService(),
)

# sfds.clear_caches()

In [5]:
cards_df = sfds.get_cards_from_bulk_data('gameplay_columns')
cards_df.shape

(38233, 23)

In [6]:
def get_type_to_subtype_dict(fpath: str) -> dict[str, list[str]]:
    with open("./data/types_to_subtypes.json", 'r') as f:
        return json.loads(f.read())

def get_types_from_type_line(type_line: str, valid_card_types: list[str]) -> list[str]:
    return sorted([x for x in valid_card_types if x.lower() in type_line.lower()])

def get_subtypes_from_type_line(type_line: str, type_to_subtype_dict: dict[str, list[str]]) -> list[str]:
    subtypes = []
    for k, v in type_to_subtype_dict.items():
        if k.lower() not in type_line.lower():
            continue
        for subtype in v:
            if subtype.lower() not in type_line.lower():
                continue
            subtypes.append(subtype.lower())
    return sorted(subtypes)

def get_supertypes_from_type_line(type_line: str) -> list[str]:
    supertypes = ["basic", "legendary", "ongoing", "snow", "world"]
    return sorted([x for x in supertypes if x in type_line.lower()])

def select_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return df[[x for x in columns if x in df.columns]]

def transform_type_line(df: pd.DataFrame, type_to_subtype_dict: dict[str, list[str]]) -> pd.DataFrame:
    df['is_split_card'] = df['name'].apply(lambda x: "//" in x)
    df['supertypes'] = df['type_line'].apply(get_supertypes_from_type_line)
    df['types'] = df['type_line'].apply(functools.partial(get_types_from_type_line, valid_card_types=type_to_subtype_dict.keys()))
    df['subtypes'] = df['type_line'].apply(functools.partial(get_subtypes_from_type_line, type_to_subtype_dict=type_to_subtype_dict))
    # df = df.drop(columns=['type_line'])
    return df

commander_df = cards_df.copy()
commander_df = commander_df \
    .pipe(functools.partial(transform_type_line, type_to_subtype_dict=get_type_to_subtype_dict("./data/type_to_subtypes.json"))) \
    # .pipe(functools.partial(select_columns, columns=['name', 'oracle_text'])).head(5)

commander_df.head()

,id,oracle_id,name,cmc,mana_cost,color_identity,colors,type_line,oracle_text,power,toughness,keywords,game_changer,defense,hand_modifier,life_modifier,loyalty,produced_mana,reserved,is_commander_legal,price_usd,penny_rank,edhrec_rank,is_split_card,supertypes,types,subtypes
0,a471b306-4941-4e46-a0cb-d92895c16f8a,00037840-6089-42ec-8c5c-281f9f474504,"Nissa, Worldsoul Speaker",4.0,{3}{G},[G],[G],Legendary Creature — Elf Druid,"Landfall — Whenever a land you control enters,...",3,3,[Landfall],False,NaN,NaN,NaN,NaN,NaN,False,True,0.25,NaN,7716.0,False,[legendary],[creature],"[druid, elf]"
1,6b7baaa7-67f2-4902-a4c1-761d58cce1d4,000492bf-7eaa-4939-a51c-4eef74e4c1d1,Mine Security,2.0,{1}{R},[R],[R],Creature — Kavu Soldier,"Trample\nWhen this creature enters, conjure a ...",3,1,"[Trample, Conjure]",False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN,NaN,False,[],[creature],"[kavu, soldier]"
2,86bf43b1-8d4e-4759-bb2d-0b2e03ba7012,0004ebd0-dfd6-4276-b4a6-de0003e94237,Static Orb,3.0,{3},[],[],Artifact,"As long as this artifact is untapped, players ...",NaN,NaN,[],False,NaN,NaN,NaN,NaN,NaN,False,True,14.87,NaN,5720.0,False,[],[artifact],[]
3,7050735c-b232-47a6-a342-01795bfd0d46,0006faf6-7a61-426c-9034-579f2cfcfa83,Sensory Deprivation,1.0,{U},[U],[U],Enchantment — Aura,Enchant creature\nEnchanted creature gets -3/-0.,NaN,NaN,[Enchant],False,NaN,NaN,NaN,NaN,NaN,False,True,0.05,NaN,28507.0,False,[],[enchantment],[]
4,9c5e9b5f-6e71-4afc-abbd-ba4c788a3240,00078ea3-0462-4a6e-b7b1-25fea012b2b7,Kavaron Consumed,5.0,{3}{R}{R},[R],[R],Sorcery,You may put an artifact or creature card from ...,NaN,NaN,[],False,NaN,NaN,NaN,NaN,NaN,False,False,NaN,NaN,NaN,False,[],[sorcery],[]
